# Router Baseline

The goal of this notebook is to train and inspect a baseline guarded query router for the GQR benchmark.

The workflow focuses on:
- defining the four benchmark labels
- loading and normalizing in-domain and out-of-domain text data
- training an embedding-based Law/Finance/Health classifier
- rejecting uncertain queries as out-of-domain
- evaluating the final router with ID accuracy, OOD accuracy, and the GQR score
- saving the trained artifacts for later notebook and benchmark usage

The reusable implementation lives in `src/router`. This notebook keeps the experiment readable while sharing the same helper modules used by the rest of the project.

## 1. Environment

The first cell prepares the notebook environment. It finds the project root, adds `src` to the Python path, configures logging, and sets the cache directory used for downloaded datasets and embedding models.

On Windows, the notebook uses a short cache path (`C:/hf-cache/router`) to avoid Hugging Face lock-file paths exceeding the platform path-length limit.

In [3]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if os.name == "nt":
    os.environ["ROUTER_CACHE_DIR"] = r"C:\hf-cache\router"
else:
    os.environ["ROUTER_CACHE_DIR"] = str(PROJECT_ROOT / "data" / "cache")

from router.cache import configure_project_cache

DATA_CACHE_DIR = configure_project_cache()

try:
    import datasets.config as datasets_config

    datasets_config.HF_HOME = str(DATA_CACHE_DIR / "huggingface")
    datasets_config.HF_DATASETS_CACHE = str(DATA_CACHE_DIR / "huggingface" / "datasets")
except ImportError:
    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

{
    "project_root": PROJECT_ROOT,
    "data_cache_dir": DATA_CACHE_DIR,
    "router_cache_dir_env": os.environ["ROUTER_CACHE_DIR"],
}

{'project_root': PosixPath('/Users/tomas/fiit/nsiete_projekt3'),
 'data_cache_dir': PosixPath('/Users/tomas/fiit/nsiete_projekt3/data/cache'),
 'router_cache_dir_env': '/Users/tomas/fiit/nsiete_projekt3/data/cache'}

The output confirms the project root and the active cache location. On Windows the cache should point to `C:/hf-cache/router`; on other platforms it stays under the project. 

In [4]:
cache_overview = {
    "HF_HOME": os.environ.get("HF_HOME"),
    "HF_DATASETS_CACHE": os.environ.get("HF_DATASETS_CACHE"),
    "HF_HUB_CACHE": os.environ.get("HF_HUB_CACHE"),
    "SENTENCE_TRANSFORMERS_HOME": os.environ.get("SENTENCE_TRANSFORMERS_HOME"),
}
cache_overview

{'HF_HOME': '/Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface',
 'HF_DATASETS_CACHE': '/Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface/datasets',
 'HF_HUB_CACHE': '/Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface/hub',
 'SENTENCE_TRANSFORMERS_HOME': '/Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface/sentence-transformers'}

## 2. Labels

The benchmark uses four integer labels. The first three labels are in-domain routing targets, while label `3` is reserved for out-of-domain queries that should be rejected by the router.

In [5]:
from router.labels import (
    ALL_LABELS,
    DOMAIN_TO_LABEL,
    ID_LABELS,
    LABEL_TO_DOMAIN,
    OOD,
    normalize_label,
)

label_rows = [
    {"label": label, "domain": LABEL_TO_DOMAIN[label]}
    for label in ALL_LABELS
]
pd.DataFrame(label_rows)

,label,domain
0,0,law
1,1,finance
2,2,health
3,3,ood


This table is the complete label space used by the router. During training, labels `0`, `1`, and `2` are treated as in-domain classes; label `3` is used only when the model decides that a query does not belong to any supported domain.

In [6]:
examples = ["law", "fintech", "healthcare", "other", "2"]
pd.DataFrame(
    {"input": value, "normalized_label": normalize_label(value)}
    for value in examples
)

,input,normalized_label
0,law,0
1,fintech,1
2,healthcare,2
3,other,3
4,2,2


The normalization helper accepts both integer labels and common text aliases. This is useful when local files use domain names such as `fintech`, `medical`, or `other` instead of the benchmark integers.

## 3. Configuration

All experiment settings are collected in one place. This makes it easy to switch between official GQR data and a local CSV/JSON/Parquet file, reduce sample sizes for a quick run, or try a different embedding model.

In [7]:
MODEL_DIR = PROJECT_ROOT / "artifacts" / "router"

TRAIN_PATH = None
VALID_PATH = None
OOD_VALID_PATH = None

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
THRESHOLD = 0.55
TARGET_ID_RECALL = 0.95
BATCH_SIZE = 64

MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_OOD_VALID_SAMPLES = 1000
SKIP_OOD_VALIDATION = False

For quick smoke tests, set `MAX_TRAIN_SAMPLES` and `MAX_VALID_SAMPLES` to small values. For a full baseline run, leave them as `None` so the loader keeps all available examples.

## 4. Load Data

The training split contains the three in-domain classes: law, finance, and health. If the official GQR package cannot load its configured data, the project falls back to public Hugging Face datasets for the same three domains.

At this stage we only prepare train and validation splits. OOD examples are added later during evaluation if the validation split does not already contain them.

In [8]:
from router.data import (
    DatasetSplit,
    load_builtin_ood_validation_dataset,
    load_gqr_ood_test_dataset,
    load_gqr_train_dataset,
    load_tabular_dataset,
)


def split_summary(name: str, split: DatasetSplit) -> dict[str, object]:
    counts = pd.Series(split.labels).map(LABEL_TO_DOMAIN).value_counts().to_dict()
    return {
        "split": name,
        "rows": len(split.texts),
        "law": counts.get("law", 0),
        "finance": counts.get("finance", 0),
        "health": counts.get("health", 0),
        "ood": counts.get("ood", 0),
    }


if TRAIN_PATH is not None:
    train_split = load_tabular_dataset(TRAIN_PATH)
    valid_split = load_tabular_dataset(VALID_PATH) if VALID_PATH else train_split
else:
    train_split, valid_split = load_gqr_train_dataset()

train_split = train_split.limit(MAX_TRAIN_SAMPLES)
valid_split = valid_split.limit(MAX_VALID_SAMPLES)

pd.DataFrame(
    [
        split_summary("train", train_split),
        split_summary("validation", valid_split),
    ]
).set_index("split")

00:24:54 | INFO | Using project cache directory: /Users/tomas/fiit/nsiete_projekt3/data/cache
00:24:54 | INFO | Loading official GQR training and validation datasets
00:24:54 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/dim/law_stackexchange_prompts/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
00:24:54 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
00:24:54 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/dim/law_stackexchange_prompts/6fd43bed8b8b288b2825937cb7fd3931cf14d802/README.md "HTTP/1.1 200 OK"
00:24:54 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/dim/law_stackexchange_prompts/resolve/6fd43bed8b8b288b2825937cb7fd3931cf14d802/law_stackexchange_prompts.py "HTTP/1.1 404 Not Found"
00:24:55 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/dim/law_stackexchange_promp

,rows,law,finance,health,ood
split,,,,,
train,28800,9611,9635,9554,0
validation,7200,2402,2409,2389,0


The summary table checks whether the loaded data has the expected domain coverage. A normal in-domain training run should contain law, finance, and health examples, while OOD rows may still be zero at this point.

## 5. Train

The baseline router has two parts. First, a SentenceTransformer model converts each query into an embedding. Then a logistic-regression classifier predicts one of the three in-domain labels.

OOD detection is handled by a confidence threshold. If the best in-domain probability is below the threshold, the router returns label `3` instead of forcing a domain prediction.

In [9]:
from router.model import DomainRouter

router = DomainRouter(
    embedding_model=EMBEDDING_MODEL,
    threshold=THRESHOLD,
)

router.fit(
    train_split.texts,
    train_split.labels,
    valid_texts=valid_split.texts,
    valid_labels=valid_split.labels,
    target_id_recall=TARGET_ID_RECALL,
    batch_size=BATCH_SIZE,
)

router.threshold

00:25:05 | INFO | Preparing training data: 28800 total rows
00:25:05 | INFO | Training domain classifier on 28800 in-domain rows
00:25:08 | INFO | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
00:25:08 | INFO | Using sentence-transformers cache: /Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface/sentence-transformers
00:25:08 | INFO | No device provided, using mps
00:25:08 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
00:25:08 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
00:25:09 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
00:25:09 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-ca

0.8905394779413293

The value displayed above is the calibrated rejection threshold. Higher thresholds usually reject more queries as OOD, while lower thresholds route more queries into one of the three supported domains.

## 6. Evaluate

Evaluation follows the GQR setup: in-domain examples measure routing accuracy, while OOD examples measure rejection accuracy. The final GQR score is the harmonic mean of these two values, so the router needs to do both tasks well.

In [10]:
from router.metrics import confusion_counts, evaluate_router


def load_ood_validation_split() -> DatasetSplit:
    if SKIP_OOD_VALIDATION:
        return DatasetSplit(texts=[], labels=[])

    if OOD_VALID_PATH is not None:
        return load_tabular_dataset(OOD_VALID_PATH).limit(MAX_OOD_VALID_SAMPLES)

    try:
        return load_gqr_ood_test_dataset().limit(MAX_OOD_VALID_SAMPLES)
    except Exception as exc:
        print(f"Falling back to built-in OOD examples: {type(exc).__name__}: {exc}")
        return load_builtin_ood_validation_dataset(
            max_samples=MAX_OOD_VALID_SAMPLES,
        )


if any(label == OOD for label in valid_split.labels):
    report_split = valid_split
else:
    report_split = valid_split.extend(load_ood_validation_split())

predictions = router.predict(report_split.texts, batch_size=BATCH_SIZE)
scores = evaluate_router(report_split.labels, predictions)

pd.DataFrame(
    [
        {
            "id_accuracy": scores.id_accuracy,
            "ood_accuracy": scores.ood_accuracy,
            "gqr_score": scores.gqr_score,
            "rows": len(report_split.texts),
            "ood_rows": sum(label == OOD for label in report_split.labels),
        }
    ]
)

00:27:15 | INFO | Using project cache directory: /Users/tomas/fiit/nsiete_projekt3/data/cache
00:27:15 | INFO | Loading official GQR OOD test dataset
00:27:15 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/Arsive/toxicity_classification_jigsaw "HTTP/1.1 200 OK"
00:27:15 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/Arsive/toxicity_classification_jigsaw/tree/main?recursive=false&expand=false "HTTP/1.1 200 OK"
00:27:16 | INFO | HTTP Request: GET https://huggingface.co/datasets/Arsive/toxicity_classification_jigsaw/resolve/main/val_dataset.csv "HTTP/1.1 307 Temporary Redirect"
00:27:16 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Arsive/toxicity_classification_jigsaw/dd6e43f4b089378aa82439da98150c400675e863/val_dataset.csv?%2Fdatasets%2FArsive%2Ftoxicity_classification_jigsaw%2Fresolve%2Fmain%2Fval_dataset.csv=&etag=%22cefd8b508bcf6c6c64798da0922cdf50140377de%22 "HTTP/1.1 206 Partial Content"
00:27:16 | INFO | HTTP Request: 

Cannot load dkhate dataset. Skipping it.
Error details:```401 Client Error. (Request ID: Root=1-69fbc047-011ec8ca3e1daeb4201c9e63;b89b11f2-98b2-4894-8473-d7563b68ffb8)

Cannot access gated repo for url https://huggingface.co/datasets/DDSC/dkhate/resolve/main/data/test-00000-of-00001.parquet.
Access to dataset DDSC/dkhate is restricted. You must have access to it and be authenticated to access it. Please log in.
```
Please check if you are logged in to Hugging Face Hub.
You can do this by running `huggingface-cli login` in your terminal.
Ensure you have access to the dataset: https://huggingface.co/datasets/DDSC/dkhate


00:27:20 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/stanfordnlp/web_questions "HTTP/1.1 200 OK"
00:27:20 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/Stanford/web_questions/tree/main/data?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
00:27:20 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/stanfordnlp/web_questions/tree/main/data?recursive=false&expand=false "HTTP/1.1 200 OK"
00:27:20 | INFO | HTTP Request: GET https://huggingface.co/datasets/Stanford/web_questions/resolve/main/data/test-00000-of-00001.parquet "HTTP/1.1 307 Temporary Redirect"
00:27:20 | INFO | HTTP Request: GET https://huggingface.co/datasets/stanfordnlp/web_questions/resolve/main/data/test-00000-of-00001.parquet "HTTP/1.1 302 Found"
00:27:20 | INFO | HTTP Request: GET https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdd236468d709f181ff5/3a3e6e605fd18efe4fbce3139b696c7bed0af05b42c4b164f6f8315f945eeec3?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Content-

,id_accuracy,ood_accuracy,gqr_score,rows,ood_rows
0,0.946806,0.637,0.761603,8200,1000


The score table separates in-domain routing quality from OOD rejection quality. The GQR score is low if either side is weak, so it is more informative here than plain accuracy over the combined dataset.

In [11]:
confusion_rows = [
    {
        "true_label": true,
        "true_domain": LABEL_TO_DOMAIN[true],
        "predicted_label": pred,
        "predicted_domain": LABEL_TO_DOMAIN[pred],
        "count": count,
    }
    for (true, pred), count in sorted(
        confusion_counts(report_split.labels, predictions).items()
    )
]

pd.DataFrame(confusion_rows)

,true_label,true_domain,predicted_label,predicted_domain,count
0,0,law,0,law,2236
1,0,law,1,finance,6
2,0,law,3,ood,160
3,1,finance,0,law,10
4,1,finance,1,finance,2238
5,1,finance,2,health,4
6,1,finance,3,ood,157
7,2,health,1,finance,3
8,2,health,2,health,2343
9,2,health,3,ood,43


The confusion table shows where the router makes mistakes. In-domain confusions indicate routing errors between supported domains, while predictions of `ood` for an in-domain true label indicate over rejection.

## 7. Inspect Predictions

A few hand-written queries make the model behavior easier to interpret than aggregate metrics alone. The examples below cover the three in-domain areas and one clearly unrelated query.

In [12]:
sample_queries = [
    "Can my landlord keep my deposit after I move out?",
    "How should I diversify my retirement portfolio?",
    "What are common symptoms of iron deficiency?",
    "How do I bake sourdough bread at home?",
]

prediction_details = router.predict_with_scores(sample_queries, batch_size=BATCH_SIZE)
pd.DataFrame(
    {
        "query": query,
        "label": prediction.label,
        "domain": LABEL_TO_DOMAIN[prediction.label],
        "confidence": round(prediction.confidence, 4),
    }
    for query, prediction in zip(sample_queries, prediction_details)
)

00:29:13 | INFO | Encoding 4 texts with batch_size=64


,query,label,domain,confidence
0,Can my landlord keep my deposit after I move out?,3,ood,0.8808
1,How should I diversify my retirement portfolio?,1,finance,0.9999
2,What are common symptoms of iron deficiency?,3,ood,0.6426
3,How do I bake sourdough bread at home?,1,finance,0.9994


The prediction table shows both the final label and the confidence used for thresholding. A query can have a likely in-domain class internally but still be returned as `ood` when confidence is below the calibrated threshold.

## 8. Save And Reload

After training, the router artifacts are saved to `artifacts/router`. This includes the fitted classifier metadata and, once the embedding model has been loaded, a local copy of the embedding model for more reliable offline prediction.

In [13]:
router.save(MODEL_DIR)
reloaded_router = DomainRouter.load(MODEL_DIR)

{
    "model_dir": str(MODEL_DIR),
    "smoke_prediction": reloaded_router.predict_one(
        "How do I appeal a denied insurance claim?"
    ),
}

00:31:19 | INFO | Saving router artifacts to /Users/tomas/fiit/nsiete_projekt3/artifacts/router
00:31:19 | INFO | Saving embedding model to /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model
00:31:19 | INFO | Saving model to /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]
00:31:19 | INFO | Using local embedding model from /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model
00:31:19 | INFO | Loading embedding model: /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model
00:31:19 | INFO | Using sentence-transformers cache: /Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface/sentence-transformers
00:31:19 | INFO | No device provided, using mps
00:31:19 | INFO | Loading SentenceTransformer model from /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 34319.46it/s]
00:31:19 | INFO | Emb

{'model_dir': '/Users/tomas/fiit/nsiete_projekt3/artifacts/router',
 'smoke_prediction': 1}

## 9. Scoring Function

The scoring helpers provide a benchmark compatible interface. They hide model loading behind a small wrapper so external evaluation code can call a simple function that maps text to a label.

In [14]:
from router.scoring import scoring_function, scoring_function_batch

single_label = scoring_function(
    "Can I sue for breach of contract?",
    model_dir=MODEL_DIR,
)
batch_labels = scoring_function_batch(
    [
        "Can I sue for breach of contract?",
        "What is a reasonable ETF allocation?",
    ],
    model_dir=MODEL_DIR,
    batch_size=BATCH_SIZE,
)

{"single_label": single_label, "batch_labels": batch_labels}

00:31:25 | INFO | Using local embedding model from /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model
00:31:25 | INFO | Loading embedding model: /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model
00:31:25 | INFO | Using sentence-transformers cache: /Users/tomas/fiit/nsiete_projekt3/data/cache/huggingface/sentence-transformers
00:31:25 | INFO | No device provided, using mps
00:31:25 | INFO | Loading SentenceTransformer model from /Users/tomas/fiit/nsiete_projekt3/artifacts/router/embedding_model.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19372.79it/s]
00:31:25 | INFO | Embedding model loaded
00:31:25 | INFO | Encoding 1 texts with batch_size=64
00:31:26 | INFO | Encoding 2 texts with batch_size=64


{'single_label': 3, 'batch_labels': [3, 1]}

These helpers are the cleanest interface for benchmark evaluation. The single text function is convenient for simple tests, while the batch variant is more efficient for larger scoring runs.

## 10. Experiments

In the second notebook, we plan these experiments:

- Test SetFit with few-shot examples and contrastive fine-tuning.
- Tune the confidence threshold for the ID/OOD tradeoff.
- Compare `4`, `8`, and `16` shots per class.
- Compare MiniLM, MultiQA, and MPNet embedding models.
- Compare the best SetFit setup with the full-data baseline.
- Check whether full MPNet SetFit can run within CPU memory.